<a href="https://colab.research.google.com/github/semarie-y/aptcommunity/blob/main/notebooks/APT_Community_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 아파트 커뮤니티 시설 공급 패턴 분석
### 데이터마이닝 | 양세미 (2024720536)
---
**연구 주제**: 서울시 신축 아파트 단지의 커뮤니티 시설 공급 패턴을 분석하고, 인구 구조·세대 구성·주변 상권 데이터를 활용한 수요 간접 예측 모델 구축  
**GitHub**: https://github.com/semarie-y/aptcommunity

---
## 목차
1. 환경 설정 및 데이터 로드
2. 데이터 전처리 — 시설 분류 (핵심 과제)
3. 데이터 JOIN — 인구·세대·상권 연결
4. 탐색적 데이터 분석 (EDA)


---
## 1. 환경 설정 및 데이터 로드

In [ ]:
# 필요 라이브러리 설치
!pip install shap -q
!apt-get install -y fonts-nanum -qq

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import seaborn as sns
import shap
import re
import warnings
warnings.filterwarnings('ignore')

# 한글 폰트 설정
fe = fm.FontEntry(fname='/usr/share/fonts/truetype/nanum/NanumGothic.ttf', name='NanumGothic')
fm.fontManager.ttflist.insert(0, fe)
plt.rcParams.update({'font.family': 'NanumGothic', 'axes.unicode_minus': False})

print('환경 설정 완료')

In [ ]:
# ── 데이터 로드 ──────────────────────────────────────────────
# Google Drive 사용 시 아래 주석 해제
# from google.colab import drive
# drive.mount('/content/drive')

# 파일 경로 (GitHub에서 클론하거나 직접 업로드)
fac_raw = pd.read_csv('복지분양시설_단지단위.csv', encoding='utf-8')
pop     = pd.read_csv('서울_인구데이터_10년단위_3년치.csv', encoding='utf-8')
hh      = pd.read_csv('서울_세대수_디테일전체.csv', encoding='utf-8')
sc      = pd.read_csv('소상공인 상권정보_서울_필터.csv', encoding='utf-8')

print(f'시설 원본: {len(fac_raw):,}행')
print(f'인구 데이터: {len(pop):,}행 ({pop["연도"].unique()})')
print(f'세대 데이터: {len(hh):,}행')
print(f'상권 데이터: {len(sc):,}행')

---
## 2. 데이터 전처리 — 시설 분류 (핵심 과제)

세움터 건축HUB 복지분양시설 데이터의 '기타시설' 컬럼은 자유 형식 텍스트로 입력되어 있어  
동일 시설이 '피트니스', '휘트니스', 'GX룸', '체력단련실' 등 다양한 표기로 기재됨.  
→ **키워드 사전(200개+) 기반 텍스트 매칭** 전략 채택

In [ ]:
# ── Step 1: 시설 텍스트 통합 컬럼 생성 ──────────────────────
fac_raw['시설통합'] = fac_raw['기타시설'].fillna('') + ' ' + \
                     fac_raw['복리분양시설종류코드명'].fillna('')

# ── Step 2: 키워드 사전 정의 (8개 유형) ─────────────────────
keyword_map = {
    '운동시설':      ['피트니스','헬스','휘트니스','FITNESS','GX','체력단련',
                    '운동시설','운동장','체육시설','주민운동','실내운동','실외운동',
                    '옥내운동','옥외운동','생활체육','배드민턴','탁구','테니스',
                    '골프','수영장','볼링','요가','필라테스','스쿼시','배구',
                    '정구','게이트볼','스포츠','운동실','미니짐','레슨룸','필라테스룸'],

    '교육_학습시설': ['작은도서관','도서관','독서실','문고','도서실','북카페',
                    '라이브러리','스터디','학습','독서','서재','정보문화실',
                    '숲속도서관','어린이도서관','에듀커뮤니티','방과후교실',
                    '창의센터','동아리','합주실','교육시설'],

    '키즈시설':      ['어린이집','키즈','맘스','다함께돌봄','돌봄센터','돌봄',
                    '어린이놀이터','유아놀이터','실내놀이터','유아용놀이터',
                    '놀이방','공동육아','보육','영유아','장난감은행','유치원',
                    '청소년','실내어린이','어린이놀이방','맘까페','맘스카페',
                    '맘스라운지','키즈라운지','키즈클럽','키즈존','키즈카페',
                    '키즈스테이션','부모대기','학부모대기','종합보육','기타아동','아동관련'],

    '시니어시설':    ['경로당','노인정','노인','시니어','어르신','재가노인',
                    '노인교실','경로'],

    '공유오피스_회의':['공유오피스','코워킹','워크스테이션','워킹라운지',
                    '스터디라운지','스터디카페','미팅룸','프라이빗 오피스',
                    '창업지원','비즈니스','회의실','회의장','입주자회의',
                    '주민회의','주민집회','집회소','다목적실','다목적홀',
                    '다목적룸','다목적회의','동호회실','멀티룸','멀티프로그램'],

    '라운지_휴게':   ['라운지','휴게시설','휴게소','휴게실','휴게공간',
                    '스카이라운지','스카이커뮤니티','카페라운지','북라운지',
                    '커뮤니티라운지','티하우스','주민카페','게스트하우스',
                    '게스트룸','게스트','공유주방','공동취사','공동주방',
                    '식당','연회장','카페테리아','전망카페','라운지카페',
                    '웰컴라운지','커뮤니티 룸','커뮤니티룸','맘스스테이션',
                    '바이크스테이션','파티룸','시네마','컬쳐라운지','스카이카페'],

    '사우나_목욕':   ['사우나','목욕','공중목욕탕','탈의실','샤워','락커'],

    '편의_생활시설': ['세탁','코인세탁','세탁실','빨래방','세탁소','창고',
                    '세대창고','공유창고','계절창고','택배','무인택배',
                    '자전거','전기차 충전','슈퍼마켓','편의점','소매점','공방'],
}

# ── Step 3: 키워드 매칭으로 0/1 분류 ────────────────────────
def classify(text):
    text = str(text)
    return {col: int(any(kw in text for kw in keywords))
            for col, keywords in keyword_map.items()}

classified = fac_raw['시설통합'].apply(classify).apply(pd.Series)
fac_classified = pd.concat([fac_raw[['대지위치','건물명']], classified], axis=1)

print('분류 완료')
print(classified.sum().to_string())

In [ ]:
# ── Step 4: 단지 단위 피벗 (건물명 기준 groupby → max) ──────
cat_cols = list(keyword_map.keys())

agg_dict = {'대지위치': 'first'}
for col in cat_cols:
    agg_dict[col] = 'max'

fac_unit = fac_classified.groupby('건물명').agg(agg_dict).reset_index()

# ── Step 5: 운동시설=1 → 사우나_목욕=1 가정 적용 ─────────────
# 운동시설 보유 단지는 샤워·목욕시설이 병행된다고 가정
fac_unit.loc[fac_unit['운동시설'] == 1, '사우나_목욕'] = 1

print(f'단지 수: {len(fac_unit)}개')
print('\n시설별 보유 현황:')
for col in cat_cols:
    n   = fac_unit[col].sum()
    pct = n / len(fac_unit) * 100
    print(f'  {col:15s}: {int(n):4d}개 ({pct:.1f}%)')

---
## 3. 데이터 JOIN — 인구·세대·상권 연결

In [ ]:
# ── 인구 파생변수 (2024년 기준) ──────────────────────────────
pop24 = pop[pop['연도'] == 2024].copy()
pop24['ratio_child']   = pop24['0~9세']   / pop24['총인구수']
pop24['ratio_elderly'] = (pop24['60~69세'] + pop24['70~79세'] +
                          pop24['80~89세'] + pop24['90~99세']) / pop24['총인구수']
pop24['ratio_young']   = pop24['20~29세'] / pop24['총인구수']
pop24['ratio_middle']  = (pop24['30~39세'] + pop24['40~49세']) / pop24['총인구수']

# ── 세대 파생변수 ────────────────────────────────────────────
hh['ratio_1person'] = hh['1인세대'] / hh['전체세대']
hh['ratio_2person'] = hh['2인세대'] / hh['전체세대']   # 신혼부부 간접 지표
hh['ratio_3plus']   = (hh['3인세대'] + hh['4인세대'] + hh['5인세대'] +
                       hh['6인세대'] + hh['7인세대'] + hh['8인세대'] +
                       hh['9인세대'] + hh['10인이상세대']) / hh['전체세대']

# ── 시설 데이터에서 구·동 추출 ───────────────────────────────
def extract_gu(addr):
    m = re.search(r'서울특별시 (\S+구)', str(addr))
    return m.group(1) if m else None

def extract_dong(addr):
    m = re.search(r'(\S+동\d*가?|\S+가\d*)', str(addr))
    return m.group(1) if m else None

fac_unit['구'] = fac_unit['대지위치'].apply(extract_gu)
fac_unit['동'] = fac_unit['대지위치'].apply(extract_dong)

print('파생변수 생성 완료')

In [ ]:
# ── LEFT JOIN ─────────────────────────────────────────────────
pop_cols = ['구','동','총인구수','ratio_child','ratio_elderly','ratio_young','ratio_middle']
hh_cols  = ['구','동','전체세대','ratio_1person','ratio_2person','ratio_3plus']

df = fac_unit.merge(pop24[pop_cols], on=['구','동'], how='left')
df = df.merge(hh[hh_cols],           on=['구','동'], how='left')

matched = df['총인구수'].notna().sum()
print(f'1차 매칭: {matched}/{len(df)} ({matched/len(df)*100:.1f}%)')

# ── 구 평균으로 결측 보완 ────────────────────────────────────
num_cols = ['ratio_child','ratio_elderly','ratio_young','ratio_middle',
            'ratio_1person','ratio_2person','ratio_3plus']

gu_mean = df.groupby('구')[num_cols].mean()
for col in num_cols:
    df[col] = df.apply(
        lambda r: gu_mean.loc[r['구'], col] if pd.isna(r[col]) else r[col], axis=1
    )

# ── 잔여 결측 제거 ───────────────────────────────────────────
df = df.dropna(subset=['ratio_child']).reset_index(drop=True)

print(f'최종 분석 단지 수: {len(df)}개')
print(f'결측치: {df[num_cols].isnull().sum().sum()}개')

# 저장
df.to_csv('apt_final_v2.csv', index=False, encoding='utf-8-sig')
print('apt_final_v2.csv 저장 완료')

---
## 4. 탐색적 데이터 분석 (EDA)

In [ ]:
TARGETS  = ['운동시설','교육_학습시설','키즈시설','시니어시설','공유오피스_회의','라운지_휴게']
FEATURES = ['ratio_child','ratio_elderly','ratio_young','ratio_middle',
            'ratio_1person','ratio_2person','ratio_3plus']
LABELS   = {
    'ratio_child':   '아동비율(0~9세)',
    'ratio_elderly': '고령비율(60세+)',
    'ratio_young':   '청년비율(20~29세)',
    'ratio_middle':  '중년비율(30~49세)',
    'ratio_1person': '1인가구비율',
    'ratio_2person': '2인가구비율',
    'ratio_3plus':   '3인이상비율',
}

In [ ]:
# ── 차트 1: 시설별 보유율 ────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 5))
rates  = {t: df[t].mean()*100 for t in TARGETS}
colors = ['#4A90D9','#5BAD6F','#E8883A','#9B59B6','#E74C3C','#1ABC9C']
bars   = ax.bar(rates.keys(), rates.values(), color=colors, alpha=0.85, width=0.6)
for bar, val in zip(bars, rates.values()):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.8,
            f'{val:.1f}%', ha='center', fontsize=11, fontweight='bold')
ax.set_ylabel('보유율 (%)')
ax.set_title('서울시 아파트 커뮤니티 시설 유형별 보유율 (997개 단지)', fontsize=14, fontweight='bold')
ax.set_ylim(0, 80)
ax.axhline(50, color='gray', linestyle='--', linewidth=0.8, alpha=0.5)
sns.despine()
plt.tight_layout()
plt.savefig('chart1_facility_rate.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── 차트 2: 피처 분포 히스토그램 ────────────────────────────
fig, axes = plt.subplots(2, 4, figsize=(16, 7))
axes = axes.flatten()
for i, col in enumerate(FEATURES):
    axes[i].hist(df[col]*100, bins=30, color='#4A90D9', edgecolor='white', alpha=0.85)
    axes[i].axvline(df[col].mean()*100, color='red', linestyle='--', linewidth=1.5,
                    label=f'평균 {df[col].mean()*100:.1f}%')
    axes[i].set_title(LABELS[col], fontsize=11, fontweight='bold')
    axes[i].set_xlabel('%')
    axes[i].legend(fontsize=9)
    sns.despine(ax=axes[i])
axes[-1].axis('off')
plt.suptitle('인구·세대 변수 분포 (동별 기준)', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('chart2_feature_dist.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── 차트 3: 상관관계 히트맵 ─────────────────────────────────
corr_cols = FEATURES + TARGETS
corr      = df[corr_cols].corr()

labels_kor = [LABELS[f] for f in FEATURES] + TARGETS
corr.index   = labels_kor
corr.columns = labels_kor

fig, ax = plt.subplots(figsize=(12, 8))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, vmin=-1, vmax=1,
            linewidths=0.5, ax=ax, annot_kws={'size': 9})
ax.add_patch(plt.Rectangle((7, 0), 6, 7, fill=False,
             edgecolor='#1F4E79', linewidth=2.5))
ax.set_title('피처 × 타겟 상관관계 히트맵\n(파란 박스: 인구·세대 변수 ↔ 시설 유형)',
             fontsize=13, fontweight='bold', pad=15)
plt.xticks(rotation=30, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig('chart3_corr_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── 차트 4: 시설 보유 여부별 인구 특성 비교 ─────────────────
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()
for idx, target in enumerate(TARGETS):
    ax    = axes[idx]
    means = df.groupby(target)[FEATURES].mean() * 100
    means.columns = [LABELS[f] for f in FEATURES]
    means = means.T
    x     = np.arange(len(means))
    width = 0.38
    ax.bar(x-width/2, means[0], width, label='미보유', color='#B0BEC5', alpha=0.9)
    ax.bar(x+width/2, means[1], width, label='보유',   color='#4A90D9', alpha=0.9)
    diffs = abs(means[1] - means[0])
    for i, (diff, val) in enumerate(zip(diffs, means[1])):
        if diff > 2:
            sign = '+' if means[1].iloc[i] > means[0].iloc[i] else '-'
            ax.text(i+width/2, val+0.3, f'{sign}{diff:.1f}',
                    ha='center', fontsize=8, color='#E74C3C', fontweight='bold')
    ax.set_title(target, fontsize=12, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(means.index, rotation=35, ha='right', fontsize=8)
    ax.set_ylabel('%')
    ax.legend(fontsize=9)
    sns.despine(ax=ax)
plt.suptitle('시설 보유 여부별 인구·세대 특성 비교 (빨간 수치: 2%p 이상 차이)',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('chart4_facility_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── 차트 5: 구별 시설 보급률 히트맵 ─────────────────────────
gu_rate = df.groupby('구')[TARGETS].mean().round(3) * 100
gu_rate = gu_rate.sort_values('키즈시설', ascending=False)

fig, ax = plt.subplots(figsize=(11, 11))
sns.heatmap(gu_rate, annot=True, fmt='.0f', cmap='YlOrRd',
            linewidths=0.5, ax=ax, annot_kws={'size': 10},
            cbar_kws={'label': '보급률 (%)', 'shrink': 0.8})
ax.set_title('서울 자치구별 커뮤니티 시설 보급률 (%) — 키즈시설 기준 정렬',
             fontsize=13, fontweight='bold', pad=15)
ax.set_xlabel('시설 유형', fontsize=11)
ax.set_ylabel('자치구', fontsize=11)
plt.xticks(rotation=20, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig('chart5_gu_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── 차트 6: 핵심 가설 산점도 ────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
scatter_pairs = [
    ('ratio_1person', '공유오피스_회의', '1인가구 비율', 'Q1: 1인가구↑ → 공유오피스 공급↑'),
    ('ratio_child',   '키즈시설',       '아동 비율',    'Q2: 아동비율↑ → 키즈시설 공급↑'),
    ('ratio_elderly', '시니어시설',     '고령 비율',    'Q3: 고령비율↑ → 시니어시설 공급↑'),
]
for ax, (feat, target, xlabel, title) in zip(axes, scatter_pairs):
    colors_map = df[target].map({0: '#B0BEC5', 1: '#4A90D9'})
    ax.scatter(df[feat]*100, df[target] + np.random.uniform(-0.15, 0.15, len(df)),
               c=colors_map, alpha=0.5, s=18)
    for val, color, label in [(0,'#B0BEC5','미보유'), (1,'#4A90D9','보유')]:
        mean_val = df[df[target]==val][feat].mean()*100
        ax.axvline(mean_val, color=color, linestyle='--', linewidth=1.5,
                   label=f'{label} 평균: {mean_val:.1f}%')
    ax.set_xlabel(f'{xlabel} (%)', fontsize=11)
    ax.set_yticks([0, 1])
    ax.set_yticklabels(['미보유', '보유'], fontsize=10)
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.legend(fontsize=9)
    sns.despine(ax=ax)
plt.suptitle('핵심 가설 검증: 인구 특성 vs 시설 공급 여부', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('chart6_hypothesis_scatter.png', dpi=150, bbox_inches='tight')
plt.show()